In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import json
import os

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'Doc': '#4C72B0', 'Img': '#DD8452', 'Movie': '#55A868',
    'Rec': '#C44E52', 'BGM': '#8172B3'
}
DPI = 300

# Load LOO evaluation data
with open('../DI_TriCHEF/results/20260425_012939_loo_eval_doc.json', 'r') as f:
    loo_data = json.load(f)

# Load Phase Ridge data
with open('../publication/paper/_phase_ridge_results.json', 'r') as f:
    ridge_data = json.load(f)

In [ ]:
# fig06_alpha_sweep_metrics.png - Alpha tuning line chart

alphas = [0.2, 0.35, 0.5, 0.65, 1.0]

dense_r1  = [0.720, 0.700, 0.620, 0.353, 0.000]
dense_r5  = [0.907, 0.880, 0.793, 0.553, 0.000]
dense_mrr = [0.797, 0.778, 0.699, 0.436, 0.000]

hybrid_r1  = [0.833, 0.807, 0.760, 0.607, 0.033]
hybrid_r5  = [0.900, 0.900, 0.880, 0.800, 0.080]
hybrid_mrr = [0.869, 0.850, 0.814, 0.695, 0.053]

fig, ax = plt.subplots(figsize=(12, 7))

# Shaded recommended alpha range
ax.axvspan(0.15, 0.35, alpha=0.12, color='gold', label='권장 Alpha 범위 (0.15–0.35)')

# Dense lines (solid)
ax.plot(alphas, dense_r1,  color='#4C72B0', linewidth=2,   marker='o', label='Dense  R@1')
ax.plot(alphas, dense_r5,  color='#55A868', linewidth=2,   marker='s', label='Dense  R@5')
ax.plot(alphas, dense_mrr, color='#C44E52', linewidth=2,   marker='^', label='Dense  MRR')

# Dense+Sparse lines (dashed)
ax.plot(alphas, hybrid_r1,  color='#4C72B0', linewidth=2, marker='o', linestyle='--', label='Dense+Sparse  R@1')
ax.plot(alphas, hybrid_r5,  color='#55A868', linewidth=2, marker='s', linestyle='--', label='Dense+Sparse  R@5')
ax.plot(alphas, hybrid_mrr, color='#C44E52', linewidth=2, marker='^', linestyle='--', label='Dense+Sparse  MRR')

# Highlight optimal point: alpha=0.2, Dense+Sparse R@1=83.3%
ax.scatter([0.2], [0.833], color='gold', s=200, zorder=10, marker='*',
           edgecolors='black', linewidths=0.8, label='최적점 (alpha=0.2, R@1=83.3%)')
ax.annotate(
    'alpha=0.2\nR@1=83.3%',
    xy=(0.2, 0.833), xytext=(0.28, 0.78),
    fontsize=10,
    arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='gold')
)

ax.set_xlabel('Alpha', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Alpha 파라미터 튜닝 (Doc 도메인 LOO 평가)', fontsize=14, fontweight='bold')
ax.set_xticks(alphas)
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()

out_path = os.path.join(os.path.dirname(os.path.abspath('fig06_alpha_loo_eval.ipynb')), 'fig06_alpha_sweep_metrics.png')
plt.savefig(out_path, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# fig06_phase_ridge_separation.png - Phase Ridge match vs null theta separation

# Domain data: [q05, q25, q50, q75, q95]
domains = ['Img', 'Doc', 'Movie', 'Rec']
domain_colors = [COLORS['Img'], COLORS['Doc'], COLORS['Movie'], COLORS['Rec']]

# Theta quantiles in degrees (match distributions)
match_quantiles = {
    'Img':   [0.15, 0.35, 0.59, 0.92, 1.40],
    'Doc':   [0.22, 0.52, 0.88, 1.45, 2.30],
    'Movie': [0.20, 0.50, 0.85, 1.40, 2.20],
    'Rec':   [0.25, 0.58, 0.97, 1.55, 2.50],
}

# Theta quantiles in degrees (null distributions)
null_quantiles = {
    'Img':   [0.45, 0.90, 1.42, 2.10, 3.20],
    'Doc':   [0.90, 1.80, 3.22, 4.80, 7.00],
    'Movie': [1.00, 2.20, 4.03, 6.00, 8.50],
    'Rec':   [0.55, 1.10, 2.07, 3.20, 4.80],
}

# Separation (q50 null - q50 match)
separations = {'Img': 0.8, 'Doc': 2.3, 'Movie': 3.2, 'Rec': 1.1}

fig, ax = plt.subplots(figsize=(14, 7))

n_domains = len(domains)
group_width = 0.6
box_half = group_width / 4.0  # half-width of each box

for i, (domain, color) in enumerate(zip(domains, domain_colors)):
    x_center = i * 1.5  # space groups apart
    x_match = x_center - group_width / 4.0
    x_null  = x_center + group_width / 4.0

    mq = match_quantiles[domain]
    nq = null_quantiles[domain]

    # --- Match box (blue-ish) ---
    m_q05, m_q25, m_q50, m_q75, m_q95 = mq
    # IQR box
    rect_m = plt.Rectangle(
        (x_match - box_half / 2, m_q25), box_half, m_q75 - m_q25,
        facecolor='#5B9BD5', alpha=0.75, edgecolor='#2E5F8A', linewidth=1.5
    )
    ax.add_patch(rect_m)
    # Median line
    ax.plot([x_match - box_half / 2, x_match + box_half / 2], [m_q50, m_q50],
            color='#1A3A5C', linewidth=2)
    # Whiskers
    ax.plot([x_match, x_match], [m_q05, m_q25], color='#2E5F8A', linewidth=1.2)
    ax.plot([x_match, x_match], [m_q75, m_q95], color='#2E5F8A', linewidth=1.2)
    ax.plot([x_match - box_half / 4, x_match + box_half / 4], [m_q05, m_q05],
            color='#2E5F8A', linewidth=1.2)
    ax.plot([x_match - box_half / 4, x_match + box_half / 4], [m_q95, m_q95],
            color='#2E5F8A', linewidth=1.2)

    # --- Null box (red-ish) ---
    n_q05, n_q25, n_q50, n_q75, n_q95 = nq
    rect_n = plt.Rectangle(
        (x_null - box_half / 2, n_q25), box_half, n_q75 - n_q25,
        facecolor='#E07070', alpha=0.75, edgecolor='#8B1A1A', linewidth=1.5
    )
    ax.add_patch(rect_n)
    # Median line
    ax.plot([x_null - box_half / 2, x_null + box_half / 2], [n_q50, n_q50],
            color='#5C0000', linewidth=2)
    # Whiskers
    ax.plot([x_null, x_null], [n_q05, n_q25], color='#8B1A1A', linewidth=1.2)
    ax.plot([x_null, x_null], [n_q75, n_q95], color='#8B1A1A', linewidth=1.2)
    ax.plot([x_null - box_half / 4, x_null + box_half / 4], [n_q05, n_q05],
            color='#8B1A1A', linewidth=1.2)
    ax.plot([x_null - box_half / 4, x_null + box_half / 4], [n_q95, n_q95],
            color='#8B1A1A', linewidth=1.2)

    # --- Separation annotation ---
    sep = separations[domain]
    y_arrow_top = n_q50
    y_arrow_bot = m_q50
    x_ann = x_null + box_half / 2 + 0.05
    ax.annotate(
        '', xy=(x_ann, y_arrow_top), xytext=(x_ann, y_arrow_bot),
        arrowprops=dict(arrowstyle='<->', color='black', lw=1.5)
    )
    ax.text(x_ann + 0.04, (y_arrow_top + y_arrow_bot) / 2,
            f'+{sep:.1f}°',
            fontsize=9, va='center', color='black',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

    # Domain label on x-axis
    ax.text(x_center, -0.5, domain, ha='center', va='top', fontsize=12,
            fontweight='bold', color=color)

# Legend proxies
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#5B9BD5', edgecolor='#2E5F8A', alpha=0.75, label='Match 분포'),
    Patch(facecolor='#E07070', edgecolor='#8B1A1A', alpha=0.75, label='Null 분포'),
]
ax.legend(handles=legend_elements, fontsize=11, loc='upper left')

# Axes formatting
ax.set_xlim(-0.6, (n_domains - 1) * 1.5 + 0.7)
ax.set_ylim(-0.8, 10.5)
ax.set_xticks([])
ax.set_ylabel('Theta (도, degrees)', fontsize=12)
ax.set_title('Phase Ridge: Match vs Null Theta 분리도 (도메인별)', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()

out_path2 = os.path.join(os.path.dirname(os.path.abspath('fig06_alpha_loo_eval.ipynb')), 'fig06_phase_ridge_separation.png')
plt.savefig(out_path2, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path2}')